In [65]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,departure_time,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class,delay_minutes
0,4227,02:58,788.12,103.94,2,sunny,Fri,0,3,61.463731,intermediate,11
1,4676,05:19,408.42,96.60,7,sunny,Sat,0,13,125.058439,premium,17
2,800,18:44,440.24,92.54,1,sunny,Sun,0,12,178.797255,standard,0
3,3671,06:27,345.01,104.57,5,sunny,Sat,0,11,137.304807,standard,5
4,4193,22:09,729.77,82.12,7,sunny,Wed,0,6,193.314124,premium,27


In [66]:
#preprocessing - ordinal encoding by hand, label encoding and scaling

df_train.weather = df_train.weather.map({"sunny": 0, "rain": 1, "snow": 2, "fog": 3})
df_train.comfort_class = df_train.comfort_class.map({"standard": 0, "intermediate": 1, "premium": 2})

df_test.weather = df_test.weather.map({"sunny": 0, "rain": 1, "snow": 2, "fog": 3})
df_test.comfort_class = df_test.comfort_class.map({"standard": 0, "intermediate": 1, "premium": 2})

from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
df_train.weekday = enc.fit_transform(df_train.weekday)
df_test.weekday = enc.transform(df_test.weekday)

df_train.head()

,SampleID,departure_time,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class,delay_minutes
0,4227,02:58,788.12,103.94,2,0,0,0,3,61.463731,1,11
1,4676,05:19,408.42,96.60,7,0,2,0,13,125.058439,2,17
2,800,18:44,440.24,92.54,1,0,3,0,12,178.797255,0,0
3,3671,06:27,345.01,104.57,5,0,2,0,11,137.304807,0,5
4,4193,22:09,729.77,82.12,7,0,6,0,6,193.314124,2,27


In [67]:
df_train.columns

Index(['SampleID', 'departure_time', 'distance_km', 'avg_speed_kmh',
       'num_stops', 'weather', 'weekday', 'special_events', 'num_cars',
       'ticket_price', 'comfort_class', 'delay_minutes'],
      dtype='object')

In [68]:
features = ['distance_km', 'avg_speed_kmh', 'num_stops', 'weather', 'weekday', 'special_events', 'num_cars', 'ticket_price', 'comfort_class']
target = 'delay_minutes'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class
0,1.719000,1.034528,-0.841728,-0.719037,-1.481059,-0.322107,-1.625539,-0.941430,0.401239
1,-0.057173,0.721845,0.905502,-0.719037,-0.484887,-0.322107,1.293106,0.282567,1.691396
2,0.091676,0.548889,-1.191174,-0.719037,0.013199,-0.322107,1.001241,1.316870,-0.888918
3,-0.353794,1.061366,0.206610,-0.719037,-0.484887,-0.322107,0.709377,0.518271,-0.888918
4,1.446049,0.104997,0.905502,-0.719037,1.507458,-0.322107,-0.749946,1.596274,1.691396


In [69]:
import torch

num_epochs = 15
batch_size = 64
learning_rate = 0.02

device = torch.device("cuda")
device

device(type='cuda')

In [70]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [71]:
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

X_train = torch.as_tensor(X_train.to_numpy(), dtype=torch.float32)
y_train = torch.as_tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)

X_test = torch.as_tensor(X_test.to_numpy(), dtype=torch.float32)
y_test = torch.as_tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size)

train_dataset[0][0]

tensor([ 1.6154, -0.5289, -1.5406,  0.4305,  0.5113, -0.3221, -0.1662,  0.8165,
        -0.8889])

In [72]:
import torch.nn as nn
import torch.optim.lr_scheduler as lr_scheduler

model = nn.Sequential(
    nn.Linear(9, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.2)

model

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=32, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=32, out_features=1, bias=True)
)

In [73]:
def train(epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print("Epoch ", epoch)
    print("Training Loss ", total_loss/len(train_loader))

def test(epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    print("Validation Loss ", total_loss/len(test_loader))

In [74]:
for i in range(1, num_epochs+1):
    train(i)
    test(i)

Epoch  1
Training Loss  80.1695767211914
Validation Loss  35.78293712322529
Epoch  2
Training Loss  46.06745594024658
Validation Loss  35.67046722998986
Epoch  3
Training Loss  45.41379425048828
Validation Loss  35.89433948810284
Epoch  4
Training Loss  43.57395595550537
Validation Loss  33.84238683260404
Epoch  5
Training Loss  43.539023094177246
Validation Loss  32.53802446218637
Epoch  6
Training Loss  41.98971858978271
Validation Loss  33.38022232055664
Epoch  7
Training Loss  40.15359165191651
Validation Loss  32.935760644766
Epoch  8
Training Loss  41.724231491088865
Validation Loss  31.55722133929913
Epoch  9
Training Loss  40.74277431488037
Validation Loss  32.63607949476976
Epoch  10
Training Loss  40.72771457672119
Validation Loss  30.647543100210335
Epoch  11
Training Loss  38.3841784286499
Validation Loss  30.852586012620193
Epoch  12
Training Loss  38.025558624267575
Validation Loss  31.257188356839695
Epoch  13
Training Loss  39.920908012390136
Validation Loss  30.8572841

In [75]:
X_final = torch.as_tensor(X_final.to_numpy(), dtype=torch.float32)

final_dataset = TensorDataset(X_final)
final_loader = DataLoader(dataset=final_dataset, batch_size=batch_size)

final_dataset[0][0]

tensor([ 0.2321,  0.5195,  1.6044, -0.7190, -0.9830,  3.1046, -0.7499,  1.5341,
        -0.8889])

In [76]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        outputs = model(inputs[0]).numpy()
        preds.extend(outputs)

preds[:10]

[array([29.606302], dtype=float32),
 array([7.246098], dtype=float32),
 array([2.495784], dtype=float32),
 array([13.597405], dtype=float32),
 array([4.3335476], dtype=float32),
 array([11.625693], dtype=float32),
 array([41.47976], dtype=float32),
 array([14.501567], dtype=float32),
 array([24.361914], dtype=float32),
 array([8.009468], dtype=float32)]

In [77]:
preds = [int(pred[0]) for pred in preds]
preds[:10]

[29, 7, 2, 13, 4, 11, 41, 14, 24, 8]

In [78]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "delay_minutes": preds
})

df_preds

,SampleID,delay_minutes
0,1501,29
1,2586,7
2,2653,2
3,1055,13
4,705,4
...,...,...
995,4711,19
996,2313,19
997,3214,16
998,2732,12


In [79]:
df_preds.to_csv("submission.csv", index=False)